# #195 seed2024 — ConvNeXt-Large@384 RA all (Kaggle)

**前置**
1. 私有 Dataset：只上传 `cache_all_cloud.zip`（MD5 `0ce27fd91870284ce304b2bed29c10b4`），不要解压后上传 63k 小文件。
2. Accelerator：GPU T4 x2；Internet：On。
3. 把本 notebook 上传到 Kaggle：Code → New Notebook → File → Import Notebook。

**用法**
- 依次运行 Cell 1、2、3、4；Cell 5 训练 8.5h 后 timeout 是预期行为。
- 每次会话结束前点 **Save Version**；下一会话把上一轮 Output 作为 Input，再运行 Cell 5，会自动找 `resume.pt` 续跑。
- 重复到 `log.csv` 出现 epoch 16；再运行 Cell 6 生成 test TTA4，下载 CSV 回帖 #195。


In [ ]:
%%bash
set -e
python - <<'PY'
import csv, sys
from pathlib import Path
root = Path('/kaggle/input/datasets/jiahaochen7711/glaucoma/cache_all')
repo = Path('/kaggle/working/glaucoma_first/glaucoma_first')
print('ROOT', root, 'exists', root.is_dir())
for s in ['train', 'valid', 'test']:
    p = root / s
    print(s, len(list(p.glob('*.jpg'))) if p.is_dir() else 'NO DIR')
for split in ['train', 'valid', 'test']:
    f = repo / (split + '.csv')
    if not f.exists():
        print('missing csv', f)
        continue
    rows = list(csv.DictReader(f.open(encoding='utf-8-sig')))
    miss = []
    for r in rows:
        stem = Path(r['Image_name']).stem
        if not any((root / s / (stem + '.jpg')).exists() for s in ['train', 'valid', 'test']):
            miss.append(stem)
    print(split, len(rows), 'missing', len(miss), miss[:5])
    if miss:
        sys.exit('%s missing images: %d; use cache_all_cloud.zip instead' % (split, len(miss)))
PY
cd /kaggle/working
if [ ! -d glaucoma_first ]; then git clone https://github.com/dongmingkun/glaucoma_first.git; fi
cd glaucoma_first
git pull || true
pip install -q timm scikit-learn pandas pillow
python tools/check_193_env.py --repo . --cache-root /kaggle/input/datasets/jiahaochen7711/glaucoma/cache_all --work-dir /kaggle/working


In [ ]:
%%bash
cd /kaggle/working/glaucoma_first
python train.py --arch convnext_base --img-size 256   --cache-root /kaggle/input/datasets/jiahaochen7711/glaucoma/cache_all --train-on-all   --epochs 1 --batch-size 4 --accum 2 --seed 2024   --loader workers --workers 2   --max-steps-per-epoch 5 --save-every-steps 3   --out-dir /kaggle/working/runs/selftest_s2024 --tag selftest_s2024


In [ ]:
%%bash
cd /kaggle/working/glaucoma_first
python train.py --arch convnext_base --img-size 256   --cache-root /kaggle/input/datasets/jiahaochen7711/glaucoma/cache_all --train-on-all   --epochs 2 --batch-size 4 --accum 2 --seed 2024   --loader workers --workers 2   --max-steps-per-epoch 5 --save-every-steps 3   --resume-from /kaggle/working/runs/selftest_s2024/resume.pt   --out-dir /kaggle/working/runs/selftest_s2024 --tag selftest_s2024


In [ ]:
%%bash
cd /kaggle/working/glaucoma_first
python train.py --arch convnext_large --img-size 384   --cache-root /kaggle/input/datasets/jiahaochen7711/glaucoma/cache_all --train-on-all   --epochs 1 --batch-size 8 --accum 6   --lr 5e-5 --head-lr 2.5e-4 --pretrained 1 --randaugment 2 9 --seed 2024   --loader workers --workers 2 --max-steps-per-epoch 20   --out-dir /kaggle/working/runs/smoke8_s2024 --tag smoke8_s2024


## 正式训练（每次会话跑 8.5h）

Cell 5 会自动查找上一轮 Output 里的 `resume.pt`：
- 第一次没有 resume，从 epoch 1 开始；
- 之后每轮先 **Save Version**，下一会话把上一轮 Output 作为 Input，再运行本 cell；
- `timeout 30600` 退出是预期；结束后确认 `resume.pt` 和 `log.csv` 再 Save Version。


In [ ]:
%%bash
set +e
cd /kaggle/working/glaucoma_first
OUT=/kaggle/working/runs/cnxl_384_e16_ra_all_s2024
mkdir -p "$OUT"
RESUME=$(find /kaggle/input -path '*cnxl_384_e16_ra_all_s2024*/resume.pt' 2>/dev/null | head -1)
if [ -n "$RESUME" ]; then
  cp "$RESUME" "$OUT/resume.pt"
  echo "[session] resume from $RESUME"
  timeout 30600 python train.py --arch convnext_large --img-size 384     --cache-root /kaggle/input/datasets/jiahaochen7711/glaucoma/cache_all --train-on-all     --epochs 16 --batch-size 8 --accum 6     --lr 5e-5 --head-lr 2.5e-4 --pretrained 1 --randaugment 2 9 --seed 2024     --loader workers --workers 2 --save-every-steps 200     --resume-from "$OUT/resume.pt" --out-dir "$OUT" --tag cnxl_384_e16_ra_all_s2024
else
  echo "[session] fresh start"
  timeout 30600 python train.py --arch convnext_large --img-size 384     --cache-root /kaggle/input/datasets/jiahaochen7711/glaucoma/cache_all --train-on-all     --epochs 16 --batch-size 8 --accum 6     --lr 5e-5 --head-lr 2.5e-4 --pretrained 1 --randaugment 2 9 --seed 2024     --loader workers --workers 2 --save-every-steps 200     --out-dir "$OUT" --tag cnxl_384_e16_ra_all_s2024
fi
echo "=== resume/log ==="
ls -lh "$OUT/resume.pt" "$OUT/last.pt" 2>/dev/null
tail -5 "$OUT/log.csv" 2>/dev/null


## 推理与交付（16ep 完成后）

- 生成 test TTA4；
- 运行审计（没有 seed7_s42 ref 时去掉 `--ref`）；
- 下载 CSV，回帖 #195，不要自动提交。


In [ ]:
%%bash
cd /kaggle/working/glaucoma_first
mkdir -p /kaggle/working/oof
python infer.py --ckpt /kaggle/working/runs/cnxl_384_e16_ra_all_s2024/last.pt   --split test --tta 4 --workers 2 --batch-size 16 --use ema   --cache-root /kaggle/input/datasets/jiahaochen7711/glaucoma/cache_all   --csv-root /kaggle/working/glaucoma_first/glaucoma_first   --out /kaggle/working/oof/cnxl_384_e16_ra_all_s2024_test_tta4.csv
python tools/audit_193_seed7_csv.py   --csv /kaggle/working/oof/cnxl_384_e16_ra_all_s2024_test_tta4.csv   --test-csv /kaggle/working/glaucoma_first/glaucoma_first/test.csv   --json /kaggle/working/oof/audit_195_seed2024.json
ls -lh /kaggle/working/oof/
